# 🧠 Data Exploration: Brain MRI for Alzheimer's Classification

This notebook explores the brain MRI dataset before training.

**Goals:**
1. Load and visualize NIfTI brain volumes
2. Understand image dimensions, spacing, and intensity distributions
3. Explore the clinical labels (CN/MCI/AD distribution)
4. Verify data quality and identify potential issues


In [ ]:
# Core imports
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Medical imaging
import nibabel as nib

# Project imports
sys.path.insert(0, str(Path.cwd().parent))
from src.utils.io import load_nifti

# Settings
plt.style.use('default')
sns.set_palette('husl')
%matplotlib inline


## 1. Setup and Load Labels


In [ ]:
# Paths
DATA_DIR = Path("../data/raw/classification")
IMAGES_DIR = DATA_DIR / "images"
LABELS_CSV = DATA_DIR / "labels.csv"

# Check paths exist
print(f"Data directory: {DATA_DIR}")
print(f"  Exists: {DATA_DIR.exists()}")
print(f"Images directory: {IMAGES_DIR}")
print(f"  Exists: {IMAGES_DIR.exists()}")
print(f"Labels CSV: {LABELS_CSV}")
print(f"  Exists: {LABELS_CSV.exists()}")


In [ ]:
# Load labels
labels_df = pd.read_csv(LABELS_CSV)
print(f"Total subjects: {len(labels_df)}")
print(f"\nColumns: {list(labels_df.columns)}")
labels_df.head(10)


## 2. Class Distribution Analysis


In [ ]:
# Diagnosis distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Color scheme
colors = {'CN': '#2ecc71', 'MCI': '#f39c12', 'AD': '#e74c3c'}

# Bar chart
diagnosis_counts = labels_df['diagnosis'].value_counts()
ax = diagnosis_counts.plot(kind='bar', ax=axes[0], 
                           color=[colors.get(d, 'gray') for d in diagnosis_counts.index])
axes[0].set_title('Class Distribution', fontsize=14)
axes[0].set_xlabel('Diagnosis')
axes[0].set_ylabel('Number of Subjects')
axes[0].tick_params(axis='x', rotation=0)

# Add count labels on bars
for i, v in enumerate(diagnosis_counts):
    axes[0].text(i, v + 0.5, str(v), ha='center', fontweight='bold')

# Pie chart
axes[1].pie(diagnosis_counts, labels=diagnosis_counts.index, autopct='%1.1f%%',
            colors=[colors.get(d, 'gray') for d in diagnosis_counts.index])
axes[1].set_title('Class Proportions', fontsize=14)

plt.tight_layout()
plt.show()

print("\nClass counts:")
print(diagnosis_counts)


## 3. Load and Visualize a Brain Volume


In [ ]:
# Load first subject
sample_subject = labels_df.iloc[0]['subject_id']
sample_path = IMAGES_DIR / f"{sample_subject}.nii.gz"

print(f"Loading: {sample_path}")

# Load with nibabel for full info
nii = nib.load(sample_path)
volume = nii.get_fdata(dtype=np.float32)
affine = nii.affine
header = nii.header

print(f"\n=== Volume Information ===")
print(f"Shape: {volume.shape}")
print(f"Data type: {volume.dtype}")
print(f"Voxel spacing (mm): {header.get_zooms()}")
print(f"Value range: [{volume.min():.2f}, {volume.max():.2f}]")
print(f"Mean: {volume.mean():.2f}, Std: {volume.std():.2f}")


In [ ]:
def visualize_volume(volume, title="Brain MRI"):
    """Visualize axial, sagittal, and coronal slices of a 3D volume."""
    D, H, W = volume.shape
    d, h, w = D // 2, H // 2, W // 2
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    # Axial (top-down view)
    axes[0].imshow(volume[d, :, :], cmap='gray', origin='lower')
    axes[0].set_title(f'Axial (slice {d}/{D})')
    axes[0].axis('off')
    
    # Coronal (front view)
    axes[1].imshow(volume[:, h, :], cmap='gray', origin='lower')
    axes[1].set_title(f'Coronal (slice {h}/{H})')
    axes[1].axis('off')
    
    # Sagittal (side view)
    axes[2].imshow(volume[:, :, w], cmap='gray', origin='lower')
    axes[2].set_title(f'Sagittal (slice {w}/{W})')
    axes[2].axis('off')
    
    fig.suptitle(title, fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()

# Visualize the loaded volume
diagnosis = labels_df.iloc[0]['diagnosis']
visualize_volume(volume, title=f"Subject: {sample_subject} | Diagnosis: {diagnosis}")


## 4. Compare Volumes Across Diagnoses


In [ ]:
# Load one subject from each diagnosis
fig, axes = plt.subplots(3, 3, figsize=(15, 12))

for row, diagnosis in enumerate(['CN', 'MCI', 'AD']):
    # Get first subject with this diagnosis
    subject_id = labels_df[labels_df['diagnosis'] == diagnosis].iloc[0]['subject_id']
    
    # Load volume
    path = IMAGES_DIR / f"{subject_id}.nii.gz"
    vol = nib.load(path).get_fdata(dtype=np.float32)
    D, H, W = vol.shape
    
    # Plot three views
    axes[row, 0].imshow(vol[D//2, :, :], cmap='gray', origin='lower')
    axes[row, 0].set_title(f'{diagnosis}: Axial')
    
    axes[row, 1].imshow(vol[:, H//2, :], cmap='gray', origin='lower')
    axes[row, 1].set_title(f'{diagnosis}: Coronal')
    
    axes[row, 2].imshow(vol[:, :, W//2], cmap='gray', origin='lower')
    axes[row, 2].set_title(f'{diagnosis}: Sagittal')

for ax in axes.flat:
    ax.axis('off')

plt.suptitle('Brain MRI Comparison: CN vs MCI vs AD', fontsize=16)
plt.tight_layout()
plt.show()


## 5. Summary and Next Steps

Now that we've explored the data, the next steps are:

1. **Preprocessing**: Run `python scripts/preprocess_data.py` to normalize and resize all volumes
2. **Create splits**: Generate train/val/test splits at subject level
3. **Train model**: Run `python scripts/train_classifier.py`
